In [3]:
from urllib.request import urlopen
from PIL import Image
import timm

img = Image.open(urlopen(
    'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png'
))

model = timm.create_model(
    'timm/vit_small_patch14_dinov2.lvd142m',
    pretrained=True,
    num_classes=0,  # remove classifier nn.Linear
)
model = model.eval()

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

output = model(transforms(img).unsqueeze(0))  # output is (batch_size, num_features) shaped tensor

# or equivalently (without needing to set num_classes=0)

output = model.forward_features(transforms(img).unsqueeze(0))
# output is unpooled, a (1, 785, 768) shaped tensor

#output = model.forward_head(output, pre_logits=True)
# output is a (1, num_features) shaped tensor


In [4]:
output.shape

torch.Size([1, 1370, 384])

In [7]:
import torch

tensor = torch.rand(1, 3, 224, 224)

In [9]:
outputtensor = model.forward_features(tensor)


In [ ]:
output = model.forward_features(transforms(img).unsqueeze(0))


In [3]:
import torch.nn as nn

loss = nn.MSELoss(reduction='sum')

In [10]:
loss(output, outputtensor)

tensor(3606317., grad_fn=<MseLossBackward0>)

In [14]:
import timm
import torch
from torchvision import models, transforms

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD = [0.229, 0.224, 0.225]

 
class PerceptualLoss(torch.nn.Module):
    def __init__(self, model_name: str = "convnext_s"):
        """Initializes the PerceptualLoss class.

        Args:
            model_name: A string, the name of the perceptual loss model to use.

        Raise:
            ValueError: If the model_name does not contain "lpips" or "convnext_s".
        """
        super().__init__()
        if ("lpips" not in model_name) and ("convnext_s" not in model_name) and (
            "vgg16" not in model_name) and (
            "dino" not in model_name) and (
            "dino2" not in model_name):
            raise ValueError(f"Unsupported Perceptual Loss model name {model_name}")
        self.lpips = None
        self.convnext = None

        self.vgg16 = None
        self.transforms = None
        self.loss_weight_vgg16 = None

        # Using timm models
        self.dino = None 
        self.loss_weight_dino = None

        self.dino2 = None
        self.loss_weight_dino2 = None

        self.data_config = None
        self.transforms = None

        self.loss_weight_lpips = None
        self.loss_weight_convnext = None

        # Parsing the model name. We support name formatted in
        # "lpips-convnext_s-{float_number}-{float_number}", where the 
        # {float_number} refers to the loss weight for each component.
        # E.g., lpips-convnext_s-1.0-2.0 refers to compute the perceptual loss
        # using both the convnext_s and lpips, and average the final loss with
        # (1.0 * loss(lpips) + 2.0 * loss(convnext_s)) / (1.0 + 2.0).
        if "lpips" in model_name:
            self.lpips = LPIPS().eval()

        if "vgg16" in model_name:
            self.vgg16 = models.vgg16(pretrained=True).eval()
            self.transforms = transforms.Compose([
                                transforms.Resize(256),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                            ])

        if "convnext_s" in model_name:
            self.convnext = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1).eval()
        
        if "dino" in model_name:
            self.dino = timm.create_model('vit_base_patch8_224.dino', pretrained=True, num_classes=0).eval()
            self.data_config = timm.data.resolve_model_data_config(self.dino)
            self.transforms = timm.data.create_transform(**self.data_config, is_training=False)

        if "dino2" in model_name: 
            self.dino2 = timm.create_model('vit_base_patch8_224.dino', pretrained=True, num_classes=0).eval()
            self.data_config = timm.data.resolve_model_data_config(self.dino)
            self.transforms = timm.data.create_transform(**self.data_config, is_training=False)

        if "lpips" in model_name and "convnext_s" in model_name:
            loss_config = model_name.split('-')[-2:]
            self.loss_weight_lpips, self.loss_weight_convnext = float(loss_config[0]), float(loss_config[1])
            print(f"self.loss_weight_lpips, self.loss_weight_convnext: {self.loss_weight_lpips}, {self.loss_weight_convnext}")

        self.register_buffer("imagenet_mean", torch.Tensor(_IMAGENET_MEAN)[None, :, None, None])
        self.register_buffer("imagenet_std", torch.Tensor(_IMAGENET_STD)[None, :, None, None])

        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self, input: torch.Tensor, target: torch.Tensor):
        """Computes the perceptual loss.

        Args:
            input: A tensor of shape (B, C, H, W), the input image. Normalized to [0, 1].
            target: A tensor of shape (B, C, H, W), the target image. Normalized to [0, 1].

        Returns:
            A scalar tensor, the perceptual loss.
        """
        # Always in eval mode.
        self.eval()
        loss = 0.
        num_losses = 0.
        lpips_loss = 0.
        convnext_loss = 0.
        # Computes LPIPS loss, if available.
        if self.lpips is not None:
            lpips_loss = self.lpips(input, target)
            if self.loss_weight_lpips is None:
                loss += lpips_loss
                num_losses += 1
            else:
                num_losses += self.loss_weight_lpips
                loss += self.loss_weight_lpips * lpips_loss

        if self.convnext is not None:
            # Computes ConvNeXt-s loss, if available.
            input = torch.nn.functional.interpolate(input, size=224, mode="bilinear", align_corners=False, antialias=True)
            target = torch.nn.functional.interpolate(target, size=224, mode="bilinear", align_corners=False, antialias=True)
            pred_input = self.convnext((input - self.imagenet_mean) / self.imagenet_std)
            pred_target = self.convnext((target - self.imagenet_mean) / self.imagenet_std)
            convnext_loss = torch.nn.functional.mse_loss(
                pred_input,
                pred_target,
                reduction="mean")
                
            if self.loss_weight_convnext is None:
                num_losses += 1
                loss += convnext_loss
            else:
                num_losses += self.loss_weight_convnext
                loss += self.loss_weight_convnext * convnext_loss

        if self.vgg16 is not None:
            # Preprocessing using https://github.com/rasbt/deeplearning-models/blob/master/pytorch_ipynb/transfer/transferlearning-vgg16-cifar10-1.ipynb?utm_source=chatgpt.com
            input = self.transforms(input)
            target = self.transforms(target)
            pred_input = self.vgg16(input)
            pred_target = self.vgg16(target)
            vgg16_loss = torch.nn.functional.mse_loss(
                            pred_input,
                            pred_target,
                            reduction="mean")
            if self.loss_weight_vgg16 is None:
                num_losses += 1
                loss += vgg16_loss
            else:
                num_losses += self.loss_weight_vgg16
                loss += self.loss_weight_vgg16 * vgg16_loss
        
        if self.dino is not None:
            input = self.transforms(input)
            target = self.transforms(target)

            feat_input = self.dino.forward_features(input)
            feat_target = self.dino.forward_features(target)

            # Uncomment to explore the pooled latents/embeddings
            #feat_input = self.dino(input)
            #feat_target = self.dino(target)

            dino_loss = torch.nn.functional.mse_loss(
                feat_input,
                feat_target,
                reduction="mean")
            
            if self.loss_weight_dino is None:
                num_losses += 1
                loss += dino_loss
            else:
                num_losses += self.loss_weight_dino
                loss += self.loss_weight_dino * dino_loss
        
        if self.dino2 is not None:
            input = self.transforms(input)
            target = self.transforms(target)

            feat_input = self.dino2.forward_features(input)
            feat_target = self.dino2.forward_features(target)

            # Uncomment to explore the pooled latents/embeddings
            #feat_input = self.dino2(input)
            #feat_target = self.dino2(target)

            dino2_loss = torch.nn.functional.mse_loss(
                feat_input,
                feat_target,
                reduction="mean")
            
            if self.loss_weight_dino2 is None:
                num_losses += 1
                loss += dino2_loss
            else:
                num_losses += self.loss_weight_dino2
                loss += self.loss_weight_dino2 * dino2_loss
        
        # weighted avg.
        loss = loss / num_losses
        return loss

    def compute_gram_matrix(input):
        b, c, h, w = input.size()
        features = input.view(b, c, h * w)
        G = torch.bmm(features, features.transpose(1, 2))
        return G / (c * h * w)

In [15]:
import torch
from torch import nn

# Assuming PerceptualLoss is imported from the module, e.g.:
# from perceptual_loss_module import PerceptualLoss

def test_perceptual_loss():
    # Create dummy inputs: batch_size=2, channels=3, height=256, width=256
    input_tensor = torch.rand(2, 3, 256, 256)
    target_tensor = torch.rand(2, 3, 256, 256)
    
    # Test 1: Using only convnext_s model
    print("Test 1: model_name='convnext_s'")
    ploss = PerceptualLoss(model_name="convnext_s")
    loss_val = ploss(input_tensor, target_tensor)
    print("Loss:", loss_val.item())

    # Test 4: Using vgg16 only (should apply transforms inside)
    print("Test 4: model_name='vgg16'")
    ploss = PerceptualLoss(model_name="vgg16")
    loss_val = ploss(input_tensor, target_tensor)
    print("Loss:", loss_val.item())

    # Note: input and target should be PIL images or tensors in [0,1], transform will handle resizing
    # Here we provide dummy tensor, but the transforms in code expect PIL images so this might fail.
    # To test vgg16 properly, we should convert tensor to PIL or adjust transforms. For demo, we skip this.

    # Test 5: Using dino model
    print("Test 5: model_name='dino'")
    ploss = PerceptualLoss(model_name="dino")
    loss_val = ploss(input_tensor, target_tensor)
    print("Loss:", loss_val.item())

    print("Test 5: model_name='dino'")
    ploss = PerceptualLoss(model_name="dino2")
    loss_val = ploss(input_tensor, target_tensor)
    print("Loss:", loss_val.item())

    # Test 6: Invalid model_name should raise ValueError
    print("Test 6: model_name='invalid_model'")
    try:
        ploss = PerceptualLoss(model_name="invalid_model")
    except ValueError as e:
        print("Caught expected exception:", e)

if __name__ == "__main__":
    test_perceptual_loss()


Test 1: model_name='convnext_s'
Loss: 0.007334188558161259
Test 4: model_name='vgg16'
Loss: 0.056436311453580856
Test 5: model_name='dino'
Loss: 3.0269174575805664
Test 5: model_name='dino'
Loss: 4.684941291809082
Test 6: model_name='invalid_model'
Caught expected exception: Unsupported Perceptual Loss model name invalid_model
